# FSDP2 与可组合并行

TorchTitan 使用 DeviceMesh 描述各个 rank 在不同并行维度中的分工，并能组合 FSDP2、TP、PP、CP 和 EP。Wordle 训练先采用两卡 FSDP2；随后以上下文并行（Context Parallelism，CP）为例，看看序列继续变长时怎样扩展并行方案。


<img src="./images/fsdp2_sharding.png" alt="两卡 DeviceMesh 上的 FSDP2 分片" width="90%">

## DeviceMesh 与 FSDP2

启动配置使用 `data_parallel_replicate_size=1`、`data_parallel_shard_size=2` 和 `context_parallel_size=1`。DeviceMesh 将两个 rank 组织成 FSDP shard mesh，每个 rank 处理各自的数据并持有一部分训练状态。

在计算某一层时，FSDP2 按需聚合该层参数；反向完成后再通过 reduce-scatter 分发梯度。这样可以避免每个 rank 长期保存完整模型和完整优化器状态，同时保持标准 PyTorch 模块和参数语义。参数和优化器 offload 负责训练阶段与 rollout 阶段之间的 CPU/NPU 驻留切换。


## 一次训练计算中的分片变化

| 阶段 | FSDP2 行为 | 目的 |
| --- | --- | --- |
| 计算前 | 按需 all-gather 当前层参数 | 为当前层前向提供完整参数视图 |
| 前向后 | `reshard_after_forward=always` 重新分片 | 降低等待反向期间的参数驻留 |
| 反向中 | 计算梯度并执行 reduce-scatter | 将梯度结果归还对应 shard |
| 优化器更新 | 每个 rank 更新本地参数与状态分片 | 避免复制完整优化器状态 |

`reshard_after_forward=always` 会在前向后尽早释放完整参数占用，为反向阶段留出更多显存；代价是反向前可能再次通信，因此实际速度还要结合目标负载测量。


## CP 如何扩展长序列训练

| 并行维度 | 主要切分对象 | 主要解决的问题 | 本案例 |
| --- | --- | --- | --- |
| FSDP2 | 参数、梯度与优化器状态 | 降低单卡训练状态占用 | 启用，shard degree 为 2 |
| CP | 序列上下文 | 分摊长序列激活与注意力计算 | 不启用，degree 为 1 |
| TP | 层内张量 | 分摊单层矩阵计算 | 不启用 |
| PP | 模型层 | 分摊不同流水段 | 不启用 |
| EP | 专家参数与 token | 分摊 MoE 专家计算 | 不启用 |

CP 将同一序列的 token 分配到多个 rank。以 Ulysses CP 为例，注意力计算前通过 all-to-all 在序列维和注意力头维之间交换分工，使每个 rank 使用部分注意力头处理完整序列；计算完成后再通过反向 all-to-all 恢复原布局。

CP 会占用并行 rank，并增加 all-to-all 通信。启用前需要重新规划 FSDP、TP 等维度，确认 query/KV head 数能够被 CP degree 整除，再到目标序列长度上测量通信开销。当前使用 Qwen3-1.7B、两张 NPU，单条样本最长 5120 token，因此三步训练先把两张卡都用于 FSDP2，CP 保持为 1。


## packed 变长序列接入 CP 的条件

Wordle RL 会移除样本 padding，再将多条不同长度的序列拼接为连续 token 流。若后续在更长序列场景中启用 CP，适配层还需要处理以下契约：

1. 在 packed token 流尾部补齐到 CP degree 的整数倍；
2. 同步填充 `input_ids`、`position_ids` 和 labels；
3. 保留完整的 `VarlenMetadata` 变长序列边界元数据，使各 rank 识别相同的样本边界；
4. 在 CP 计算后以可求导方式汇集 log-prob，并移除新增 padding；
5. 按原始 offsets 恢复 jagged tensor，继续计算 GRPO loss。

可见，packed 变长数据接入 CP 不只是修改一个并行度参数，还要同步处理样本边界、补齐和结果还原。CP 更适合激活显存已经限制序列长度的场景；当前负载下，额外的填充、通信和结果汇集反而可能增加耗时，所以三步训练不引入 CP，将 `context_parallel_size` 保持为 1。


## 课后练习

### 判断题

1. （判断题）本案例使用两卡 FSDP2，并将 `context_parallel_size` 保持为 1。

2. （判断题）只要提高 CP degree，就一定能够缩短当前 Wordle 训练的单步耗时。

### 单选题

3. （单选题）Ulysses CP 在注意力计算前主要交换哪两个维度的分工？

   A. 序列与注意力头

   B. epoch 与学习率

   C. 模型层与 checkpoint

   D. prompt 与奖励函数

### 多选题

4. （多选题）packed 变长序列接入 CP 时需要完成哪些处理？

   A. 将总 token 数补齐到 CP degree 的整数倍

   B. 保留完整的序列边界元数据

   C. 汇集并移除 log-prob 中新增的 padding

   D. 修改 Wordle 奖励规则

5. （多选题）决定是否采用 CP 时，应评估哪些条件？

   A. 序列长度和激活显存是否构成瓶颈

   B. 注意力头数与 CP degree 的整除关系

   C. all-to-all 通信开销

   D. 其他并行维度的资源规划


> 完成练习后，运行下方单元格查看参考答案和解析。


In [ ]:
from pathlib import Path
import subprocess

course_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
answer_path = course_root / 'tutorials/rl_training_pipeline/06_torchtitan_npu_features/answer/06.02_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))
